# CUADERNO DE INVESTIGACIÓN

Comprobamos la conexión

In [8]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("data/sql-murder-mystery.db")
print(conn)

Leemos el informe del caso

In [14]:
query = """
SELECT description
FROM crime_scene_report
WHERE date = 20180115
  AND city = 'SQL City'
  AND type = 'murder';
"""

pd.read_sql_query(query, conn)


,description
0,Security footage shows that there were 2 witne...


Con esta pista sacmos que el primer testigo vive en la dirección más alta de esa calle y el segundo testigo se llama Annabel y vive en algún lugar de Franklin Ave.


In [15]:
query = """
SELECT *
FROM person
WHERE address_street_name = 'Northwestern Dr'
ORDER BY address_number DESC
LIMIT 1;
"""

pd.read_sql_query(query, conn)

,id,name,license_id,address_number,address_street_name,ssn
0,14887,Morty Schapiro,118009,4919,Northwestern Dr,111564949


In [20]:
query = """
SELECT *
FROM person
WHERE address_street_name = 'Franklin Ave'
  AND name LIKE '%Annabel%';
"""

pd.read_sql_query(query, conn)

,id,name,license_id,address_number,address_street_name,ssn
0,16371,Annabel Miller,490173,103,Franklin Ave,318771143


Ahora tenemos a los dos testigos, vamos a leer sus declaraciones

In [21]:
query = """
SELECT *
FROM interview
WHERE person_id IN (14887, 16371);
"""

pd.read_sql_query(query, conn)

,person_id,transcript
0,14887,I heard a gunshot and then saw a man run out. ...
1,16371,"I saw the murder happen, and I recognized the ..."


In [22]:
entrevistas = pd.read_sql_query("""
SELECT *
FROM interview
WHERE person_id IN (14887, 16371);
""", conn)

for _, fila in entrevistas.iterrows():
    print(f"\n--- Testigo {fila['person_id']} ---")
    print(fila['transcript'])


--- Testigo 14887 ---
I heard a gunshot and then saw a man run out. He had a "Get Fit Now Gym" bag. The membership number on the bag started with "48Z". Only gold members have those bags. The man got into a car with a plate that included "H42W".

--- Testigo 16371 ---
I saw the murder happen, and I recognized the killer from my gym when I was working out last week on January the 9th.


Estas entrevistas nos dan  pistas muy concretas:

Testigo 14887 — Morty Schapiro

El asesino llevaba una bolsa de Get Fit Now Gym.
El número de socio empezaba por 48Z.
Era Gold member.
El coche tenía una matrícula que incluía H42W.

Testigo 16371 — Annabel Miller

Reconoció al asesino de su gimnasio.
Lo vio entrenando el 9 de enero de 2018.

In [23]:
query = """
SELECT *
FROM get_fit_now_member
WHERE id LIKE '48Z%'
  AND membership_status = 'gold';
"""

pd.read_sql_query(query, conn)

,id,person_id,name,membership_start_date,membership_status
0,48Z7A,28819,Joe Germuska,20160305,gold
1,48Z55,67318,Jeremy Bowers,20160101,gold


Ahora tenemos dos sospechosos potenciales pero tenemos que cruzar estas pistas.
Así que ahora vamos a buscar coches cuyas matrículas contengan la combinación que vió Morty.

In [24]:
query = """
SELECT *
FROM drivers_license
WHERE plate_number LIKE '%H42W%';
"""

pd.read_sql_query(query, conn)

,id,age,height,eye_color,hair_color,gender,plate_number,car_make,car_model
0,183779,21,65,blue,blonde,female,H42W0X,Toyota,Prius
1,423327,30,70,brown,brown,male,0H42W2,Chevrolet,Spark LS
2,664760,21,71,black,black,male,4H42WR,Nissan,Altima


Ahora tenemos que cruzar estas matrículas con nuestros dos sospechosos (Joe Germuska y Jeremy Bowers)


In [25]:
query = """
SELECT 
    p.id,
    p.name,
    p.license_id,
    m.id AS membership_id,
    m.membership_status,
    dl.plate_number
FROM person p
JOIN get_fit_now_member m
    ON p.id = m.person_id
JOIN drivers_license dl
    ON p.license_id = dl.id
WHERE m.id LIKE '48Z%'
  AND m.membership_status = 'gold'
  AND dl.plate_number LIKE '%H42W%';
"""

pd.read_sql_query(query, conn)

,id,name,license_id,membership_id,membership_status,plate_number
0,67318,Jeremy Bowers,423327,48Z55,gold,0H42W2




Las dos pistas de Morty coinciden en la misma persona:

Bolsa del gimnasio → socio que empieza por 48Z y es Gold
Matrícula → contiene H42W
Jeremy Bowers cumple ambas:
Socio: 48Z55
Matrícula: 0H42W2
ID persona: 67318
.Tenemos que usar la declaración de Annabel para confirmar que Jeremy estaba en el gimnasio el 9 de enero de 2018.


In [26]:
query = """
SELECT *
FROM get_fit_now_check_in
WHERE check_in_date = 20180109
  AND membership_id IN ('48Z7A', '48Z55');
"""

pd.read_sql_query(query, conn)

,membership_id,check_in_date,check_in_time,check_out_time
0,48Z7A,20180109,1600,1730
1,48Z55,20180109,1530,1700



Jeremy Bowers sí estaba en el gimnasio el 9 de enero de 2018, justo como dijo Annabel, pero Joe Germuska también estaba allí (48Z7A, de 16:00 a 17:30).

Así que todavía tenemos que buscar una pista más que diferencie a los dos. Ahora queremos obtener los datos personales de Jeremy y comprobar que la información de las diferentes tablas encaja.


In [28]:
query = """
SELECT *
FROM interview
WHERE person_id = 67318;
"""

pd.read_sql_query(query, conn)

,person_id,transcript
0,67318,I was hired by a woman with a lot of money. I ...


In [30]:
entrevista_jeremy = pd.read_sql_query("""
SELECT transcript
FROM interview
WHERE person_id = 67318;
""", conn)

print(entrevista_jeremy["transcript"].iloc[0])

I was hired by a woman with a lot of money. I don't know her name but I know she's around 5'5" (65") or 5'7" (67"). She has red hair and she drives a Tesla Model S. I know that she attended the SQL Symphony Concert 3 times in December 2017.



 Jeremy confiesa que fue contratado por otra persona. Así que Jeremy parece ser el asesino ejecutor, pero tenemos que descubrir quién lo contrató.

La pista más específica es la última, así que vamos a buscar primero quién asistió 3 veces al concierto.

In [31]:
query = """
SELECT person_id, COUNT(*) AS veces
FROM facebook_event_checkin
WHERE event_name = 'SQL Symphony Concert'
  AND date LIKE '201712%'
GROUP BY person_id
HAVING COUNT(*) = 3;
"""

pd.read_sql_query(query, conn)

,person_id,veces
0,24556,3
1,99716,3


Nos quedan dos candidatas que cumplen la pista de haber asistido exactamente 3 veces al concierto en diciembre, vamos a cruzarlas con la tabla person para saber quiénes son.

In [32]:
query = """
SELECT *
FROM person
WHERE id IN (24556, 99716);
"""

pd.read_sql_query(query, conn)

,id,name,license_id,address_number,address_street_name,ssn
0,24556,Bryan Pardo,101191,703,Machine Ln,816663882
1,99716,Miranda Priestly,202298,1883,Golden Ave,987756388


In [33]:
query = """
SELECT *
FROM drivers_license
WHERE id = 202298;
"""

pd.read_sql_query(query, conn)

,id,age,height,eye_color,hair_color,gender,plate_number,car_make,car_model
0,202298,68,66,green,red,female,500123,Tesla,Model S




Miranda Priestly cumple todas las pistas, fue ella quién contrató a Jeremy Bowers para cometer el asesinato.
